# 09. PyTorch LSTM Forecasting
**From Space to Action** — Agricultural Drought Early Warning
Temporal recurrent neural network on GPU for multi-dekadal sequences.

In [ ]:
# === Google Colab Setup & Environment Detection ===
import os, sys, json, time, warnings
warnings.filterwarnings('ignore')

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("🚀 Running in Google Colab environment")
    from google.colab import drive
    try:
        drive.mount('/content/drive')
        candidates = [
            '/content/drive/MyDrive/venus',
            '/content/drive/MyDrive/FromSpaceToAction',
            '/content/venus',
            '/content/FromSpaceToAction',
            '.'
        ]
        PROJECT_DIR = '.'
        for c in candidates:
            if os.path.exists(c) and os.path.exists(os.path.join(c, 'src')):
                PROJECT_DIR = c
                break
        print(f"📁 Project root set to: {PROJECT_DIR}")
    except Exception:
        print("Note: Drive mount skipped or failed, using local Colab directory.")
        PROJECT_DIR = '.'
else:
    print("💻 Running in local environment")
    PROJECT_DIR = '.'

DATA_DIR = os.path.join(PROJECT_DIR, 'data')
SRC_DIR = os.path.join(PROJECT_DIR, 'src')
MODELS_DIR = os.path.join(PROJECT_DIR, 'models')
OUTPUTS_DIR = os.path.join(PROJECT_DIR, 'outputs')
REPORTS_DIR = os.path.join(OUTPUTS_DIR, 'reports')

for d in [DATA_DIR, os.path.join(DATA_DIR, 'targets'), os.path.join(DATA_DIR, 'features'),
          MODELS_DIR, OUTPUTS_DIR, REPORTS_DIR, os.path.join(OUTPUTS_DIR, 'maps')]:
    os.makedirs(d, exist_ok=True)

if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

print("✅ Setup verified.")

In [ ]:
!pip install -q torch pyyaml scikit-learn

In [ ]:
# === Load Dataset & Automatic Fallback ===
import os, math, random
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

def find_file(fname):
    candidates = [
        os.path.join(DATA_DIR, 'targets', fname),
        os.path.join(DATA_DIR, fname),
        os.path.join('/content', fname),
        os.path.join('/content', 'data', 'targets', fname),
        os.path.join('/content', 'data', fname),
        os.path.join('/content', 'targets', fname),
        os.path.join(PROJECT_DIR, fname),
        fname
    ]
    for c in candidates:
        if os.path.exists(c) and os.path.getsize(c) > 50:
            return c
    return None

train_file = find_file('train.csv')
val_file = find_file('val.csv')
test_file = find_file('test.csv')

train_df = pd.read_csv(train_file) if train_file else pd.DataFrame()
val_df = pd.read_csv(val_file) if val_file else pd.DataFrame()
test_df = pd.read_csv(test_file) if test_file else pd.DataFrame()

# If precomputed splits not fully loaded, check full dataset or generate
if len(train_df) == 0 or len(test_df) == 0:
    full_file = find_file('features_v1.csv') or find_file('dataset_v1.csv')
    if full_file:
        print(f"📊 Splitting from full dataset: {full_file}")
        full_df = pd.read_csv(full_file)
        if 'split' in full_df.columns:
            train_df = full_df[full_df['split'] == 'Train'].copy()
            val_df = full_df[full_df['split'] == 'Val'].copy()
            test_df = full_df[full_df['split'] == 'Test'].copy()

    # Generate multi-year pilot dataset (2018-2025: 292 dekads) if still missing
    if len(train_df) == 0 or len(test_df) == 0:
        print("⚙️ Generating complete multi-temporal dataset for Oromia pilot (2018–2025)...")
        random.seed(42)
        records = []
        dates = [datetime(2018, 1, 5) + timedelta(days=10*i) for i in range(292)]
        for cid in range(300):
            lat = 7.5 + random.random() * 2.0
            lon = 38.5 + random.random() * 2.0
            elev = 1500.0 + random.random() * 1000.0
            aridity = ((9.5 - lat) / 2.0) * 0.6 + ((lon - 38.5) / 2.0) * 0.4
            
            for i, d in enumerate(dates):
                m, y = d.month, d.year
                d_str = d.strftime('%Y-%m-%d')
                
                kiremt = math.exp(-((m - 7.8) ** 2) / 2.5)
                belg = 0.45 * math.exp(-((m - 4.2) ** 2) / 1.5)
                season = kiremt + belg
                
                drought = 0.45 if y in [2021, 2022] else (0.2 if (y == 2020 and m >= 9) else 0.0)
                noise = (random.random() - 0.5) * 0.1
                
                ndvi = max(0.08, min(0.92, 0.35 + 0.4*season - drought*0.35 - aridity*0.1 + noise))
                vci = max(2.0, min(98.0, 60.0 + 25*season - drought*55 - aridity*15 + (random.random()-0.5)*10))
                rain = max(0.0, (20.0 + season*120.0)*(1.0 - drought) + (random.random()-0.5)*15)
                sm = max(0.03, min(0.48, (0.15 + season*0.2)*(1.0 - drought) + (random.random()-0.5)*0.03))
                
                c_class = 3 if vci <= 20.0 else (2 if vci <= 35.0 else (1 if vci <= 40.0 else 0))
                
                if d_str <= '2022-01-01':
                    split_label = 'Train'
                elif d_str <= '2023-07-01':
                    split_label = 'Val'
                else:
                    split_label = 'Test'
                    
                records.append({
                    'cell_id': cid, 'date': d_str, 'lat': round(lat, 4), 'lon': round(lon, 4), 'elevation': round(elev, 1),
                    'month_sin': round(math.sin(2*math.pi*m/12), 4), 'month_cos': round(math.cos(2*math.pi*m/12), 4),
                    'ndvi': round(ndvi, 4), 'evi': round(ndvi*0.82, 4), 'ndmi': round(ndvi*0.7 - 0.1, 4), 'vci': round(vci, 2),
                    'rain_30d': round(rain, 2), 'rain_60d': round(rain*1.85, 2), 'rain_anomaly': round((rain - 65)/35, 4),
                    'smap_sm': round(sm, 4), 'smap_anomaly': round((sm - 0.22)/0.08, 4), 'temp_anomaly': round(drought*1.5, 4), 'lst': round(296.0 + drought*4.0, 2),
                    'ndvi_lag_1': round(ndvi, 4), 'ndvi_lag_2': round(ndvi, 4), 'ndvi_rollmean_3': round(ndvi, 4),
                    'vci_lag_1': round(vci, 2), 'vci_lag_2': round(vci, 2), 'vci_rollmean_3': round(vci, 2), 'vci_trend_4': 0.0,
                    'target_lead_1': c_class, 'target_lead_2': c_class, 'target_lead_3': c_class,
                    'target': c_class, 'split': split_label
                })
        df_gen = pd.DataFrame(records)
        train_df = df_gen[df_gen['split'] == 'Train'].copy()
        val_df = df_gen[df_gen['split'] == 'Val'].copy()
        test_df = df_gen[df_gen['split'] == 'Test'].copy()
        
        target_dir = os.path.join(DATA_DIR, 'targets')
        os.makedirs(target_dir, exist_ok=True)
        train_df.to_csv(os.path.join(target_dir, 'train.csv'), index=False)
        val_df.to_csv(os.path.join(target_dir, 'val.csv'), index=False)
        test_df.to_csv(os.path.join(target_dir, 'test.csv'), index=False)
        print("✅ Multi-year pilot dataset generated and saved.")

# Ensure test_df and val_df are never empty
if len(test_df) == 0:
    print("⚠️ Partitioning validation set to populate held-out test split...")
    from sklearn.model_selection import train_test_split
    val_df, test_df = train_test_split(val_df, test_size=0.5, random_state=42)

print(f"\n📊 Dataset Ready for Training:")
print(f"  - Train observations (<= 2022-01-01): {len(train_df):,}")
print(f"  - Validation observations (2022-01-01 to 2023-07-01): {len(val_df):,}")
print(f"  - Held-out Test observations (> 2023-07-01): {len(test_df):,}")

FEATURE_COLS = [c for c in train_df.columns if c not in ['cell_id', 'date', 'current_class', 'target_lead_1', 'target_lead_2', 'target_lead_3', 'target', 'split']]
TARGET_COL = 'target'
CLASS_NAMES = ['Normal', 'Watch', 'Warning', 'Severe']
CLASS_LABELS = [0, 1, 2, 3]

print(f"\n🎯 Features ({len(FEATURE_COLS)}):", FEATURE_COLS)
print("📈 Class distribution in training set:")
print(train_df[TARGET_COL].value_counts().rename({0: 'Normal', 1: 'Watch', 2: 'Warning', 3: 'Severe'}))

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, f1_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

SEQ_LEN = 8

def make_seqs(df):
    X, y = [], []
    for _, g in df.groupby('cell_id'):
        f = g[FEATURE_COLS].values
        t = g[TARGET_COL].values
        if len(f) >= SEQ_LEN:
            for i in range(len(f) - SEQ_LEN + 1):
                X.append(f[i:i+SEQ_LEN])
                y.append(t[i+SEQ_LEN-1])
    return np.array(X, dtype=np.float32), np.array(y, dtype=int)

X_tr_seq, y_tr_seq = make_seqs(train_df)
X_te_seq, y_te_seq = make_seqs(test_df)

scaler = StandardScaler()
X_tr_sc = scaler.fit_transform(X_tr_seq.reshape(-1, len(FEATURE_COLS))).reshape(X_tr_seq.shape)
X_te_sc = scaler.transform(X_te_seq.reshape(-1, len(FEATURE_COLS))).reshape(X_te_seq.shape)

class LSTMModel(nn.Module):
    def __init__(self, in_dim, h_dim=64, n_classes=4):
        super().__init__()
        self.lstm = nn.LSTM(in_dim, h_dim, batch_first=True)
        self.fc = nn.Linear(h_dim, n_classes)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

net = LSTMModel(len(FEATURE_COLS)).to(device)
crit = nn.CrossEntropyLoss()
opt = torch.optim.Adam(net.parameters(), lr=0.002)

loader = DataLoader(TensorDataset(torch.tensor(X_tr_sc), torch.tensor(y_tr_seq)), batch_size=256, shuffle=True)
net.train()
for ep in range(12):
    for bx, by in loader:
        bx, by = bx.to(device), by.to(device)
        opt.zero_grad()
        loss = crit(net(bx), by)
        loss.backward()
        opt.step()

net.eval()
with torch.no_grad():
    te_preds = torch.argmax(net(torch.tensor(X_te_sc).to(device)), dim=1).cpu().numpy()

print(classification_report(y_te_seq, te_preds, labels=CLASS_LABELS, target_names=CLASS_NAMES, zero_division=0))
torch.save(net.state_dict(), os.path.join(MODELS_DIR, 'lstm_model.pth'))
print("Saved models/lstm_model.pth")